# N11 — Audio Mel Classifier

Sessions 54–55. Build reproducible synthetic audio features, keep related sources in the same split, and evaluate a classifier.


In [ ]:
from __future__ import annotations
import platform, random
from pathlib import Path
import numpy as np
SEED=42
random.seed(SEED); np.random.seed(SEED)
print(platform.python_version(), Path.cwd())


In [ ]:
import librosa
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
sr=8000; t=np.arange(4000)/sr; rng=np.random.default_rng(SEED); rows=[]
for source in range(20):
    for j in range(8):
        label=(source+j)%2; freq=440 if label==0 else 880
        wave=np.sin(2*np.pi*freq*t)+rng.normal(0,.08,len(t))
        mel=librosa.feature.melspectrogram(y=wave.astype(np.float32),sr=sr,n_mels=24,n_fft=256,hop_length=128)
        db=librosa.power_to_db(mel,ref=np.max)
        rows.append((f'S{source:02d}',label,np.r_[db.mean(1),db.std(1)]))
X=np.stack([r[2] for r in rows]); y=np.array([r[1] for r in rows]); groups=np.array([r[0] for r in rows])
tr,va=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED).split(X,y,groups))
model=Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(max_iter=1000))]).fit(X[tr],y[tr])
assert not(set(groups[tr]) & set(groups[va]))
print('F1',f1_score(y[va],model.predict(X[va])))


## Worksheet
1. Define sample rate, waveform, spectrogram, and Mel scale.
2. Explain why related recordings should remain in one split.
3. State what mean/std pooling removes.
4. Name two label-preserving transformations.
5. Distinguish recognition, synthesis, and classification tasks.

## Independent transfer
Add one controlled change in noise, gain, or sample rate. Report results for each source group and explain one failure.


## Fresh-run checklist
- Restart and run all cells in order.
- Record package versions and seed.
- Confirm the group-overlap assertion.
- Add an error log and independent explanation.
